In [ ]:
import json
import math
import random
from collections import deque, namedtuple
from dataclasses import dataclass
import gymnasium as gym
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scipy.stats as stats
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from gymnasium import spaces
from sklearn.metrics import mean_squared_error, r2_score
import math, bisect

#pre-compute

Weather-corsswind and temperature Driver model- driving behavior
Payload variation - varying payloads with waypoints
Force coeffs - precompute independent of T or mass
Paux - all 5 auxilary usages powers

In [ ]:
###PRE-COMPUTE - finalized
##1 Crosswind fucntion(also temperature?)
# pure physics calculates only W1, W2, CdxAeff comes in when we get historical data or something
# outputs W1, W2 lookup table for a given height based on vehicle speeds and wind speeds

def crosswind_engine(vehicle_height, v_ref, h_ref, a1, a2,a3, wind_exp, dh, dphi_deg, v_veh_kmh, beta_limit_deg):

  #check if wind speeds and vehicle speeds are a list or not
  v_ref = np.atleast_1d(np.asarray(v_ref, dtype=float))
  v_veh_kmh = np.atleast_1d(np.asarray(v_veh_kmh, dtype=float))
  v_veh = v_veh_kmh / 3.6 # convert from km/h to m/s

  n_slices = round(vehicle_height / dh) # number of equal heights based on dh
  slice_heights = (np.arange(n_slices) + 0.5) * dh # slide heights but the midpoint of each slice, avoids 0 wind at 0m issue.
  phi = np.radians(np.arange(0, 360, dphi_deg)) # converts 0 to 360 deg to radians

  #the None in brackets only maintain the shape i.e. give each list a dimension to run along so there is not collisions, easy for numpy this way
  v_wind = v_ref[:, None, None, None] * (slice_heights[None, None, None, :] / h_ref) ** wind_exp #Calculates the wind speed at each height slice, wind_exp is how wind grows with height
  v_air_x = v_veh[None, :, None, None] + v_wind * np.cos(phi)[None, None, :, None] # wind hitting the vehicle on the x axis
  v_air_y = v_wind * np.sin(phi)[None, None, :, None] # wind component hitting the vehicle on the y axis
  v_air_sq = v_air_x**2 + v_air_y**2 #total air speed squared

  beta_deg = np.abs(np.degrees(np.arctan2(v_air_y, v_air_x))) #yaw angle the angle between vehicle direction and wind direction, abs since vehicle is symmetric
  if beta_limit_deg is not None: # sets the 10deg clamp so it stays VECTO auditable,above 10 it breaks the curve following
      beta_deg = np.minimum(beta_deg, beta_limit_deg)
  delta_CdxA = a1 * beta_deg + a2 * beta_deg**2 + a3 * beta_deg**3 # extra drag m2 caused by that yaw angle

  W1 = v_air_sq.mean(axis=(2, 3)) / v_veh**2 #how much more air is hitting than with no wind
  W2 = (delta_CdxA * v_air_sq).mean(axis=(2, 3)) / v_veh**2 #how much extra drag the yawed airflow adds in m2
  return W1, W2


In [ ]:
##2 Payload engine - finalized
# pure logic, does not care if waypoints came from vdri, typed input or gps
# writes payload only on the start row and on stop rows, everything else stays blank (NaN)
# the blanks get filled forward later so ui can edit stop rows directly

def payload_engine(stop, shape, payload_start, payload_end=None, payload_custom=None): #stop duration column, starting payload

  stop = np.asarray(stop, dtype=float) # stop time of every waypoint/row in s, 0 means no stop; this is just the stop column from vecto or custom inputs

  is_stop = stop > 0 # true where the vehicle stops
  is_stop[0] = False # first row is the depot start, its dwell is not a payload change
  n_stops = int(is_stop.sum()) # number of stops where load can change

  # one payload for the start + one after each stop
  if shape == "constant":
      values = np.full(n_stops + 1, float(payload_start)) # same payload everywhere

  elif shape == "linear":
      if n_stops == 0:
          raise ValueError("linear needs at least one stop")
      values = np.linspace(payload_start, payload_end, n_stops + 1) # equal change at every stop, hdev assumption (unsourced)

  elif shape == "custom":
      values = np.asarray(payload_custom, dtype=float) # user gives start + one value per stop
      if len(values) != n_stops + 1:
          raise ValueError(f"custom needs {n_stops + 1} values, got {len(values)}")
  else:
      raise ValueError(f"unknown shape: {shape}")

  if np.any(values < 0):
      raise ValueError("payload cannot be negative")

  payload = np.full(len(stop), np.nan) # blank payload column, same length as waypoints
  payload[0] = values[0] # start payload always on row 0 so forward fill has something to start from
  payload[is_stop] = values[1:] # payload after each stop written on that stop row
  return payload

In [ ]:
##3 Driver model
# driver model: VECTO DefaultDriverStrategy on fixed 1 s rows (HDEV deviation: VECTO uses ~0.5 s variable steps)
# pure kinematics when coast_mass is None; with coast_mass: look-ahead coasting + overspeed at a FIXED reference vehicle (HDEV assumption)
def driver_model(wp_s, wp_v, wp_stop, wp_grade,
                 vacc_v, vacc_acc, vacc_dec, dt=1.0,
                 coast_mass=None, m_red_wheels=0.0, cdxa_v=None, cdxa=None, Crr=None, beta=1.0, rho=1.188, g=9.80665,
                 drag_v=(0.0,), drag_F=(0.0,), P_max=None,
                 lac_min_v=50.0, lac_dist_factor=10.0, df_offset=2.5, df_scaling=1.5,
                 df_vtarget=((0, 48, 52, 100), (0, 0, 1, 1)), df_vdrop=((-100, 9, 11, 100), (1, 1, 0, 0)),
                 os_min_v=50.0, os_allowed=2.5, brake_margin=0.1, v_min_coast=5.0):

  # converting waypoints into lists
  s_wp = [float(x) for x in wp_s]                      # waypoint distance [m]
  v_wp = [float(x)  for x in wp_v]                # waypoint target speed [m/s] --> / 3.6 must be in parser
  stop_wp = [float(x) for x in wp_stop]                # FINAL standing time [s] (PTO already added by the parser)
  gr_wp = np.asarray(wp_grade, dtype=float)            # waypoint grade [%]
  n = len(s_wp)

  rise = np.diff(np.asarray(s_wp)) * gr_wp[:-1] / 100  # height gained between two consecutive waypoints
  alt_wp = np.concatenate(([0.0], np.cumsum(rise))) # altitude at each way point
  alt_l = alt_wp.tolist() # convert altitude into list
  gr_l = gr_wp.tolist() # convert gradient into list

  if v_wp[0] == 0 and stop_wp[0] == 0:                 # VECTO line 221: cycle starting at 0 km/h without stop gets a 1 s stop
      stop_wp[0] = 1.0

  # speed the driver aims for after passing waypoint i; after a stop it is the next waypoint's target (VECTO line 246-253)
  # (HDEV: if several stops follow each other, take the next POSITIVE target, so the truck never gets stuck at 0 km/h)
  seg_v = [0.0] * n # target speed after each waypoint
  nxt = 0.0 # next positive target speed moving back from waypoint
  for i in range(n - 1, -1, -1): #moving backwards
      seg_v[i] = v_wp[i] if v_wp[i] > 0 else nxt
      if v_wp[i] > 0:
          nxt = v_wp[i]
  # change points: only waypoints whose target differs from the segment before can trigger braking/coasting
  chg = [i for i in range(1, n) if v_wp[i] != seg_v[i-1]] # index where the current way point speed is diff from previous segment target speed
  chg_s = [s_wp[i] for i in chg] #get the distance of the waypoints at those index

  vv = [float(x) for x in vacc_v]                       # .vacc speeds in m/s --> / 3.6  goes in parser
  va = [float(x) for x in vacc_acc]              # max acceleration at each speed
  vd = [float(x) for x in vacc_dec]              # max decceleration at each speed

  coast_on = coast_mass is not None # coasting condition check if coasting mass is present
  if coast_on:
      if cdxa_v is None or cdxa is None or Crr is None: # checking if all parameters exist to calculate coasting
          raise ValueError("coasting needs cdxa_v, cdxa and Crr")  # guard only; parser gives friendly message
      m_eff = coast_mass + m_red_wheels                # effective mass incl. rotating wheels
      cv = [float(x) for x in cdxa_v]                   # reference CdxA table speeds in m/s -->reference speed into m/s / 3.6  in parser
      cc = [float(x) for x in cdxa]

  # /3.6 in parser
  lac_min = lac_min_v        # look ahead coasting only above this speed
  os_min = os_min_v          # overspeed allowed only when target is above this
  os_add = os_allowed        # how above target speed can it exceed downhill
  v_min_c = v_min_coast      # dont coast below this

  # acceleration with no motor torque at speed v on segment k [m/s²]
  def coast_acc(v, k):

      th = math.atan(gr_l[k] / 100)

      F = (rho / 2 * interp1(v, cv, cc) * v * v        # air drag, instantaneous (HDEV: VECTO uses the step average)
           + coast_mass**beta * g * Crr * math.cos(th)       # rolling resistance, mass without wheel inertia like VECTO
           + coast_mass * g * math.sin(th)             # slope force
           + interp1(v, drag_v, drag_F) )      # motor/gearbox drag while rolling

      return -F / m_eff         # divide by m_eff altho we use coast_mass because wheels add inertia not weight

  T, S, V, A, M = [], [], [], [], []    # rows: time, distance, speed, acceleration, driving mode
  t = 0.0                               #initialize start time
  s = s_wp[0]                           #initialize start position
  v = v_wp[0]                           # VECTO starts at the first waypoint's target speed (DistanceBasedDrivingCycle.cs line 107)
  s_end = s_wp[-1]                      # end position

  # standing rows at waypoint i
  def add_stop(i, t):

      for _ in range(int(round(stop_wp[i] / dt))): # for the number of dt time steps the stop happens
          T.append(t); S.append(s_wp[i]); V.append(0.0); A.append(0.0); M.append(0) #append stationary state to rows
          t += dt # increment time by the time step
      return t

  if v_wp[0] == 0:       # check if route starts at a stop
      t = add_stop(0, t)  # initial stop
      v = 0.0             # explicitly set the speed to 0

  trig = None   # active slowdown: [index, trigger distance, target speed, coasting allowed, braking started]
  max_steps = int(sum(stop_wp) / dt + (s_end - s_wp[0]) / 0.5 + 10000)  # guard against endless loops (truck must average > 0.5 m/s)

  #-----For each step
  for _ in range(max_steps):    # for all steps
      if s >= s_end - 1e-6:    # if travelled distance greater or equal to final distance break
          break
      k = bisect.bisect_right(s_wp, s) - 1             # waypoint segment we are in
      target = seg_v[k]                                # target speed for that segment
      v_os = target + os_add if target > os_min else target  # overspeed limit (VECTO IsOverspeedAllowed: > 50 km/h, +2.5 km/h)

      if trig is not None and s >= trig[1] - 1e-6:     # Check whether a previous slowdown trigger has been passed
          trig = None

      # look ahead (VECTO GetNextDrivingAction, lines 647-683)
      look = max(2 * v * dt, v * 3.6 * lac_dist_factor, 1.2 * decel_distance(v, 0.0, vv, vd) + v * dt) # max of:twice the vehicle travels in one step, distance based on speed,  the distance required to decelerate from the current speed to zero with safety factor
      j0 = bisect.bisect_right(chg_s, s)              # findin upcoming speed changes
      j1 = bisect.bisect_right(chg_s, s + look)
      best_b = None                                    # candidate with the earliest braking start
      best_c = None                                    # candidate with the earliest coasting start

      for j in range(j0, j1):     #loop thru every speed change way point in the look ahead range
          i = chg[j]
          vt = v_wp[i] + os_add if v_wp[i] > os_min else v_wp[i]  # target incl. overspeed (VECTO line 662)

          if vt >= v:           # Ignore speed changes that don't require slowing down
              continue
          b_start = s_wp[i] - (decel_distance(v, vt, vv, vd) + brake_margin) # Calculate the braking start position

          c_start = b_start   # initially coasting start is set equal to breaking start position
          if coast_on and v > lac_min:        # if coasting is on and if speed is above look ahead speed
              cd = coast_distance(v, vt, interp1(s, s_wp, alt_l), alt_l[i], s, s_wp[i],   # Calculate the coasting distance
                                  coast_mass, m_red_wheels, cv, cc, Crr, beta, rho, g, drag_v, drag_F,
                                  df_offset, df_scaling, df_vtarget, df_vdrop)

              if cd > 0:                                # if coasting distance is positive
                  c_start = s_wp[i] - cd                # calculate the coasting start position

          if best_b is None or b_start < best_b[1]:  #finds the upcoming speed change with the earliest braking start position
              best_b = (i, b_start, c_start, vt)
          if best_c is None or c_start < best_c[2]:   #finds the upcoming speed change with the earliest coasting start position
              best_c = (i, b_start, c_start, vt)

      action = None                                   # initially no driving action
      if best_b is not None:                           # VECTO: braking action, unless the earliest coasting one is the same point
          i, b_start, c_start, vt = best_b              # select earliest breaking candidate if there is at least one speed reduction coming up
          coast_ok = best_c[0] == i and c_start < b_start     # is coasting allowed for the same waypoint with breaking; must belong to same waypoint, coasting must start before breaking
          action = (i, s_wp[i], vt, coast_ok, min(b_start, c_start))    # creates a tupple; Waypoint index, Waypoint position, Target speed, Whether coasting is allowed, Earliest action start position

      if trig is None and action is not None and s + v * dt >= action[4]:  # start slowing down if the next step would pass the start point
          trig = [action[0], action[1], action[2], action[3], False]
      elif trig is not None and action is not None and action[1] != trig[1]:    #checks whether a new driving action has appeared while another slowdown is already active
          b_new = action[1] - (decel_distance(v, action[2], vv, vd) + brake_margin)     #Calculates the braking start position for the new target
          b_cur = trig[1] - (decel_distance(v, trig[2], vv, vd) + brake_margin)       #Calculates the braking start position for the existing target
          if b_new < b_cur:                            # a nearer slowdown needs earlier braking (VECTO CheckSwitchingToBraking)
              trig = [action[0], action[1], action[2], False, True]   # switch to new breaking action if new breaking is earlier

      # -- choose acceleration
      acc_max = interp1(v, vv, va)        #max acc allowed at current speed
      dec_max = interp1(v, vv, vd)        #max dec allowed at current speed

      if trig is not None:                #is there an active slowdown trigger
          i_t, s_t, v_t, coast_ok, braking = trig
          b_start = s_t - (decel_distance(v, v_t, vv, vd) + brake_margin)   # recalculate breaking start at current speed

          if 0 < v <= v_t:                             #situation 1; next target already reached; current speed > 0 and <= target speed
              a = max(dec_max, min(acc_max, (v_t - v) / dt)); mode = 2        #acceleration needed to reach target speed in 1 step limited by acc/dec limits
          elif coast_on and coast_ok and not braking and s + v * dt < b_start and v >= v_min_c:  #situation 2; coasting before braking start
              ca = coast_acc(v, k)                     # calculates the acceleration resulting from coasting
              a = min(ca, (max(target, v) - v) / dt)   # never faster than max(target, current speed)
              a = max(a, dec_max)                      # limits
              mode = 3 if a >= ca - 1e-9 else 4        # speed cap active -> braking, not coasting
          elif not braking and s < b_start:            # Situation 3: Braking starts during the current time step
              frac = (s + v * dt - b_start) / (v * dt) if v > 0 else 1.0      # fraction of the time step occurs after reaching the braking start position
              a = dec_max * min(max(frac, 0.0), 1.0); mode = 4  # factor is limited between 0 and 1
              trig[4] = True                           # breaking flag set to true, once braking, stay braking until the point
          else:                                        # Situation 4: Active braking
              rem = s_t - s                            # remaining distance to the target waypoint
              if rem < 2 * v * dt and v > v_t:         # checks whether the vehicle is very close to the target waypoint
                  a = (v_t**2 - v**2) / (2 * rem) if rem > 1e-6 else dec_max        # Adapt deceleration to the remaining distance
              else:                                    # If the vehicle is not yet within the final one or two time steps
                  a = max(dec_max, (v_t - v) / dt)     # calculates the acceleration needed to reach the target speed in one time step within limits
              mode = 4                                 # breaking
              trig[4] = True                           # set breaking flag

      else:                                            # when there is no active trigger
          if v < target - 1e-9:                        # does it need to accelerate
              a = min(acc_max, (target - v) / dt)      # calculates the acceleration needed to reach the target speed within one time step

              if P_max is not None and coast_on:       # optional motor power cap (HDEV simplification, not VECTO's motor search)
                  F_res = -coast_acc(v, k) * m_eff     # equivalent resistance force; F=ma
                  a = min(a, (P_max / max(v, 1.0) - F_res) / m_eff)   #estimates the maximum available acceleration after accounting for resistance forces
              if action is not None:                   # don't speed up so much that the next slowdown can't be braked for in time
                  room = action[1] - s - v * dt - brake_margin   # This calculates the remaining distance available after the current time step
                  v_cap = math.sqrt(max(action[2]**2 + 2 * -dec_max * max(room, 0.0), 0.0))   # maximum speed that would still allow the vehicle to reach target speed over reamining distance
                  a = max(min(a, (v_cap - v) / dt), 0.0)    # limits acceleration so that the vehicle does not exceed the calculated speed cap
              mode = 1                                # acceleration mode

          elif coast_on and target > os_min:           # at/above target with overspeed allowed: coast
              ca = coast_acc(v, k)                     # acc during coasting
              if ca >= 0:                              # coasting acc is > 0 (could be downhill)
                  a = max(dec_max, min(ca, (v_os - v) / dt))   # downhill: let it run up to target + 2.5 km/h, then hold
                  mode = 3 if a >= ca - 1e-9 else 4            # holding the speed needs brakes then switch to mode 4
              elif v + ca * dt > target:          # after rolling one step the speed still stays above target so keep rolling
                  a = ca
                  mode = 3                     # roll back down towards target
              else:
                  a = (target - v) / dt; mode = 2      # would drop below target -> hold target
          elif v > target + 1e-9:                      # current speed greater than target
              a = max(dec_max, (target - v) / dt)      # calculates the acceleration needed to return to the target speed
              mode = 4                                 # above target without overspeed ; brake to target
          else:                                        # current speed equal to target
              a = 0.0
              mode = 2                        # cruise

      # ---------- move one step
      v_new = v + a * dt                               # update new speed
      if v_new < 0:                                    # does vehicle stop inside the step
          ds = v * v / (2 * -a) if a < 0 else 0.0      # calculate actual stopping distance
          v_new = 0.0                                  # set speed to 0 ; stopped
      else:
          ds = (v + v_new) / 2 * dt                    # distance travelled in one step

      if trig is not None and trig[2] == 0 and (v_new <= 0 or s + ds >= trig[1]): #Does the vehicle need to stop, and has it reached the point where it must stop
          ds = max(trig[1] - s, 0.0)                   # arrive exactly at the stop (HDEV: VECTO shortens the step instead)
          v_new = 0.0

      T.append(t); S.append(s); V.append(v); A.append((v_new - v) / dt); M.append(mode)
      t += dt
      s += ds
      v = v_new

      if trig is not None and trig[2] == 0 and v == 0:  # arrived at a stop
          s = trig[1]                                   # set stop position as exact stop postiion target
          t = add_stop(trig[0], t)                      #adds the required stationary time at the stop waypoint
          trig = None
  else:
      raise RuntimeError("driver_model did not reach the end of the route")  # guard only

  # ---after the loop: arrays + grade from altitude
  T = np.asarray(T); S = np.asarray(S); V = np.asarray(V); A = np.asarray(A); M = np.asarray(M, dtype=int) # convert to numpy arrays
  s_next = np.append(S[1:], s)                         # distance at the end of each row
  ds_row = s_next - S                                  # distance travelled during each simulation interval
  h1 = np.interp(S, s_wp, alt_wp)                      # road altitude at the starting position of every simulation row
  h2 = np.interp(s_next, s_wp, alt_wp)                 # Calculate the altitude at the end of each row
  k_row = np.searchsorted(s_wp, S, side="right") - 1
  grade = np.where(ds_row > 1e-9, (h2 - h1) / np.where(ds_row > 1e-9, ds_row, 1.0) * 100, gr_wp[k_row])  # [%]; standing rows use the waypoint grade
  return T, S, V, A, grade, M


In [ ]:
# helper: linear interpolation for ONE number, holds end values like np.interp, but much faster inside a loop - finalized but comment
def interp1(x, xs, ys):
  if x <= xs[0]:
      return ys[0]
  if x >= xs[-1]:
      return ys[-1]
  i = bisect.bisect_right(xs, x)                                  # first table point above x
  return ys[i-1] + (ys[i] - ys[i-1]) * (x - xs[i-1]) / (xs[i] - xs[i-1])

In [ ]:
# helper: braking distance when speed goes from v_from to v_to - finalized
def decel_distance(v_from, v_to, vacc_v, vacc_dec):
  if v_to >= v_from: # check if the truck is slowing down
      return 0.0 # not slowing down so not breaking
  dist = 0.0 # no breaking distance
  for i in range(1, len(vacc_v)): # walk thru the vacc table segment by segment
      v_lo, v_hi = vacc_v[i-1], vacc_v[i] # two speeds at segment ends
      a_lo, a_hi = vacc_dec[i-1], vacc_dec[i] # corresponding deceleration
      v1 = min(v_from, v_hi) # top of overlap between breaking segment and this segment,
      v2 = max(v_to, v_lo) # bottom of overlap
      if v2 >= v1: # check if breaking segment passes thru this segment, bottom of overlap is not below the top
          continue
      if math.isclose(a_lo, a_hi): # checks if deceleration is constant at both ends of segment
          dist += (v1**2 - v2**2) / (2 * -a_lo) # constant-deceleration formula v1² − v2² = 2·a·s
      else: #if deceleration changes along the segment
          k = (a_lo - a_hi) / (v_lo - v_hi) # The slope of that line: the change in deceleration divided by the change in speed.
          d = a_lo - k * v_lo # second number of the straight line chosen so the line passes through (v_lo, a_lo).
          t = math.log((v2*k + d) / (v1*k + d)) / k # The time it takes to brake from v1 down to v2, v2*k + d is simply the deceleration at v2
          m = v1 + d / k # helper for below equation
          dist += m / k * math.exp(k * t) - d / k * t - m / k # distance travelled while the speed drops from v1 to v2 over the time t
  return dist

In [ ]:
# helper: look-ahead coasting distance, VECTO DefaultDriverStrategy.ComputeCoastingDistance (lines 721-756) - finalized
def coast_distance(v, v_target, h_now, h_target, s_now, s_target,
                   coast_mass, m_red_wheels, cdxa_v, cdxa, Crr, beta, rho, g, drag_v, drag_F,
                   df_offset, df_scaling, df_vtarget, df_vdrop):
  # speed, target speed, height now, target height, distance now, target distance, reference mass,
  #spinning wheel mass, air drag at speed v, drag area table, air drag coeff,

  m_eff = coast_mass + m_red_wheels # effective mass including mass of spinning wheels
  dE = m_eff * g * (h_now - h_target) + m_eff * (v**2 - v_target**2) / 2 # energy going from h to target h rolling down increases energy, kinetic energy lost from v to v target
  v_avg = (v + v_target) / 2 # average speed during coasting for drag lookup

  cda = interp1(v_avg, cdxa_v, cdxa) # air drag area cdxa at v avg using lookup table cdxa

  F_air = rho / 8 * cda * (v + v_target) * (v**2 + v_target**2) / v_avg #average air force over the whole roll from v to v_target
  grad = (h_target - h_now) / (s_target - s_now) # average slope between h and target h, height/dist

  F_roll = math.cos(grad) * coast_mass**beta * g * Crr # Tyre rolling resistance
  F_loss = interp1(v, drag_v, drag_F) #Read the drag force from the table at the current speed

  f_vt = interp1(v_target * 3.6, df_vtarget[0], df_vtarget[1]) #A factor from VECTO’s table based on the target speed (in km/h); 0 below 48 km/h, 1 above 52 km/h, and a straight-line blend in between.
  f_vd = interp1((v - v_target) * 3.6, df_vdrop[0], df_vdrop[1]) # A factor based on how big the speed drop is (in km/h); 1 if the drop is under 9 km/h, 0 if it is over 11 km/h, and a blend in between.
  DF = df_offset - df_scaling * f_vt * f_vd # VECTO lets the driver coast fully only for small slowdowns at high speed. For everything else, the driver lifts off later and brakes for the rest

  return dE / (DF * (F_air + F_roll + F_loss)) # Energy ÷ total force gives the distance, and dividing by DF shortens it.

In [ ]:
##4 FORCE ENGINE
## takes in rows from the driver model and calculates the force coeff per row independent of mass and temp and CdxA_eff
def force_engine(g, Crr, speed, accel, dt_s, grade, r_dyn, theta_red):

  speed = np.asarray(speed, dtype=float) # speed at start of the step
  accel = np.asarray(accel, dtype=float) # acceleration of the vehicle
  dt_s = np.asarray(dt_s, dtype=float)  # time step, s
  grade = np.asarray(grade, dtype=float) # gradient

  grade = np.arctan(grade / 100) # turns gradient in % into an angle in radians
  v1 = speed
  #v2 = speed + accel * dt_s
  v2 = np.maximum(speed + accel * dt_s, 0.0) # speed cannot go below 0, as in the driver model
  v_avg = (v1 + v2)/2  #speed in m/s

  #K = ((Crr * g * np.cos(grade)) + (g * np.sin(grade)) + accel) * v_avg
  # the K equation is the same with the added term of rotational force, if theta red and rdyn are 0 we get back the simple equation.
  k_lin = (g * np.sin(grade) + accel) * v_avg
  k_roll = Crr * g * np.cos(grade) * v_avg
  k_rot = (theta_red / r_dyn**2) * accel * v_avg if theta_red > 0 else np.zeros_like(v_avg) # no rotating mass unless given
  L = (v1 + v2) * (v1**2 + v2**2) / 8 # exact average of 0.5*v^3 over the step as VECTO CrosswindCorrectionCdxALookup

  return k_lin, k_roll, k_rot, L, v_avg # we are taking out v_avg so we can use it to look up CdxA

In [ ]:
##5.1 Electrical system - finalized
def P_aux(ES_power, PS_power, HVAC_power, SP_power, cond_power, PTO_always_on, ES_alteff=0.7, PS_alteff=1.0, HVAC_eff=1.25, SP_alteff=0.7, cond_eff=1.0, PTO_eff=1.0): #get rid of hardcoding later
  # pure: electric W per aux = input W * efficiency factor
  # works for one number or an array using numpy
  # PTO_always_on = always-on PTO only (chiller, mixer); stop PTO comes from pto_engine and is laid on rows in the consumption engine

  powers = [np.asarray(p, dtype=float) for p in (ES_power, PS_power, HVAC_power, SP_power, cond_power)] # convert first so lists work
  effs = (ES_alteff, PS_alteff, HVAC_eff, SP_alteff, cond_eff)
  PTO_always_on = np.asarray(PTO_always_on, dtype=float)

  for p in powers:
      if np.any(p < 0):
          raise ValueError("aux power cannot be negative")
  for e in effs:
      if e <= 0:
          raise ValueError("efficiency must be > 0")

  ES_power, PS_power, HVAC_power, SP_power, cond_power = (p * e for p, e in zip(powers, effs))
  PTO_always_on = PTO_always_on / PTO_eff  # undo the multiply above, then divide: shaft W / efficiency = electric W --> rewrite this better later

  return ES_power, PS_power, HVAC_power, SP_power, cond_power, PTO_always_on

In [ ]:
# returns mechanical W, goes into P_aux as SP_power gets multiplied by 0.7 for electrical W

def SP_power_helper(SP_baseline, SP_axle_shares, SP_tech_factors):

  SP_baseline = np.asarray(SP_baseline, dtype=float)         # (UF, B, S) in W, from SP-Table.csv
  SP_axle_shares = np.asarray(SP_axle_shares, dtype=float)   # one (UF, B, S) row in % per steered axle, from SP-Axles.csv
  SP_tech_factors = np.asarray(SP_tech_factors, dtype=float) # one (UF, B, S) row per steered axle, from SP-Tech.csv or custom

  if SP_axle_shares.ndim != 2 or SP_axle_shares.shape[1] != 3 or len(SP_axle_shares) == 0:
      raise ValueError("SP_axle_shares needs one (UF, B, S) row per steered axle")
  if SP_tech_factors.shape != SP_axle_shares.shape:
      raise ValueError("need one technology (UF, B, S) per steered axle") # VECTO throws here too

  parts = SP_baseline * SP_axle_shares.sum(axis=0) / 100 # each part summed over the steered axles
  parts = parts * SP_tech_factors.mean(axis=0)           # VECTO: average technology factor over all steered axles
  return float(parts.sum())                  # add the 3 parts,

In [ ]:
def HVAC_thermal(T_amb, solar, A_surface, U, N_persons, A_window, T_heat_points, cop_heat_points, T_cool_points, cop_cool_points,
                 hp_share_points, eh_share_points, fuel_share_points, max_heating_power, max_cooling_power, cab_volume, vent_rate, vent_rate_heating,
                 eh_eff=0.93, spec_vent_power=0.56, heat_per_person=(50.0, 80.0), clouding=(0.65, 0.8), T_boundary=17.0, g=0.95, solar_factor=0.25,
                 waste_heat=0.0, T_heat=18.0, T_cool=23.0, T_cool_off=17.0, Q=0.0, p=101325.0, R=287.05, cp=1005.0):
  # Tcalc just becomes the two setpoints for heating and cooling
  T_amb = np.asarray(T_amb, dtype=float)

  # step 1: run ssm total i.e. total heat balance at 18 and 23 C, and get the heating or cooling demand
  run1 = ssm_total_w(T_amb, solar, T_heat, A_surface, U, N_persons, A_window, heat_per_person, clouding, T_boundary, g, solar_factor, p, R, cp, Q)
  run2 = ssm_total_w(T_amb, solar, T_cool, A_surface, U, N_persons, A_window, heat_per_person, clouding, T_boundary, g, solar_factor, p, R, cp, Q)

  losing_heat = (run1 < 0) & (run2 < 0)
  heating_demand = np.where(losing_heat, np.abs(np.maximum(run1, run2))-waste_heat,0.0)
  heating_demand = np.maximum(heating_demand, 0.0)

  #step 2: how this heat is split between heat pump, electrical heater, and fuel heater and their caps at diff temps SSMCalculate.cs lines 215-245
  cop_heat, cop_cool, hp_share, eh_share, fuel_share = climate_lookup(T_amb, T_heat_points, cop_heat_points, T_cool_points, cop_cool_points, hp_share_points, eh_share_points, fuel_share_points)

  fuel_heat = heating_demand * fuel_share                                            # fuel heater part, not capped here
  sum_heat = np.minimum(max_heating_power, heating_demand * (hp_share + eh_share))   # cap on heat pump + electric heater only

  rest = np.where(np.isclose(fuel_share, 1.0), 1.0, 1 - fuel_share)  # heat pump + el. heater's part; 1.0 if fuel does everything
  scale = sum_heat / rest                                                    # getting a scaling factor to downside but keep ratio
  hp_heat = scale * hp_share                                                 # heat delivered by heat pump [W]
  eh_heat = scale * eh_share                                                 # heat delivered by electric heater [W]

  #step 3: heating W to what it costs the batery W
  if np.any((hp_heat > 0) & np.isnan(cop_heat)):
    raise ValueError("heat pump COP missing for this temperature")  # HDEV: VECTO silently gives 0 W here
  HP_el = np.where(hp_heat > 0, hp_heat / cop_heat, 0.0)             # heat pump: heat ÷ COP
  EH_el = eh_heat / eh_eff                                           # battery W for el heating

  #step 4: cooling
  gaining_heat = (run1 > 0) & (run2 > 0) & (T_amb >= T_cool_off)            # gaining heat at both 18 and 23 setpoint, and Tamb greater than 17
  cooling_demand = np.where(gaining_heat, np.minimum(run1, run2), 0.0)      # cooling demand / heat to remove W, smaller of the two = cool to 23C
  cooling_demand = np.minimum(cooling_demand, max_cooling_power)            # cap at AC max cooling power
  if np.any((cooling_demand > 0) & np.isnan(cop_cool)):
      raise ValueError("cooling COP missing for this temperature")
  AC_el = np.where(cooling_demand > 0, cooling_demand / cop_cool, 0.0)                # battery W for cooling

  #step 5; ventilation fan, VECTO SSMCalculate.cs lines 270-295, 376-397, 399-428; VentPower (SSMInputs.cs:88)
  ventillation_status = ((T_amb < T_cool_off) & (run1 > 0) & (run2 > 0)) | ((run1 > 0) & (run2 < 0))
  fan_rate = np.where(losing_heat, vent_rate_heating, np.where(gaining_heat | ventillation_status, vent_rate, 0.0)) # air changes per hour
  fan_el = cab_volume * fan_rate * spec_vent_power  #m³ × 1/h × Wh/m³ = W

  #step 6: total battery W for HVAC (goes into P_aux as HVAC_power with HVAC_eff = 1.0)
  HVAC_el = HP_el + EH_el + AC_el + fan_el    # heat pump + electric heater + AC + fan [W electric]
  return HVAC_el, fuel_heat                   # fuel_heat = heat from the diesel heater [W heat], not battery


In [ ]:
# this is a helper for HVAC thermal
# step 1: cabin heat balance, VECTO SSMRun.TotalW (SSMRun.cs lines 48-64)
def ssm_total_w(T_amb, solar, T_calc, A_surface, U, N_persons, A_window, heat_per_person, clouding, T_boundary, g, solar_factor, p, R, cp, Q):
#ambient Temp, Cabin temp target, surface area cabin, heat dissapation , num of people, surface area of windows, heat per person, clouding low or high, Temp switch for body heat and clouding, solar allowance of windows, soalr factor is unknown?
  T_amb = np.asarray(T_amb, dtype=float) # outside temp per weather hour [°C]
  solar = np.asarray(solar, dtype=float) # sunlight per weather hour [W/m²]
  rho = p / (R * (T_amb + 273.15)) # density of outside air

  q_wall = (T_amb - T_calc) * A_surface * U    #heat thru walls, diff in amb temp and cabin temp * surface area of cabin and heat permissivity
  q_people = N_persons * np.where(T_amb < T_boundary, heat_per_person[0], heat_per_person[1]) # heat thru people, no of people * their body heat based on amb Temp
  q_solar = solar * A_window * g * np.where(T_amb < T_boundary, clouding[0], clouding[1]) * solar_factor # heat from the sun passing thru windows based on clouding
  q_freshair = rho * cp * Q / 3600 * (T_amb - T_calc)
  return q_wall + q_people + q_solar + q_freshair

In [ ]:
#This is a helper for HVAC thermal
def climate_lookup(T_amb, T_heat_points, cop_heat_points, T_cool_points, cop_cool_points, hp_share_points, eh_share_points, fuel_share_points):
  # abm temp,5 VECTO heat points
  T_amb = np.asarray(T_amb, dtype=float)

  cop_heat = np.interp(T_amb, T_heat_points, cop_heat_points)  # heat pump heating COP per hour
  cop_cool = np.interp(T_amb, T_cool_points, cop_cool_points)  # cooling COP per hour

  dist = np.abs(T_amb[..., None] - np.asarray(T_heat_points, dtype=float))  # distance from each hour to each heating point
  idx = np.argmin(dist, axis=-1)                          # nearest point; on a tie argmin takes the first = colder

  hp_share = np.asarray(hp_share_points, dtype=float)[idx]     # heat pump part of the heat
  eh_share = np.asarray(eh_share_points, dtype=float)[idx]     # electric heater part
  fuel_share = np.asarray(fuel_share_points, dtype=float)[idx] # fuel heater part

  return cop_heat, cop_cool, hp_share, eh_share, fuel_share

In [ ]:
#PTO engine
def pto_engine(stop, is_on, stop_pto_t, stop_pto_w, drive_pto_t, drive_pto_w, stop_pto_eff, drive_pto_eff):

  stop = np.asarray(stop, dtype=float) # stop time of every waypoint/row in s, 0 means no stop; this is just the stop column from vecto or custom inputs this also includes PTO usage time
  is_on = np.array(is_on, dtype=float)       # 1 = PTO on at this waypoint, 0 = off
  stop_pto_t = np.asarray(stop_pto_t, dtype=float) #pto time profile
  stop_pto_w = np.asarray(stop_pto_w, dtype=float) #pto power profile
  drive_pto_w = np.asarray(drive_pto_w, dtype=float) #drive power profile
  drive_pto_t = np.asarray(drive_pto_t, dtype=float) #drive time profile


  is_on[0] = 0 # the first waypoint is the depot: never PTO there

  # turn each profile into one power value per second
  stop_steps = np.diff(stop_pto_t).astype(int) # how many seconds each value lasts
  stop_1s = np.repeat(stop_pto_w[:-1], stop_steps) # stop PTO, W per second

  drive_steps = np.diff(drive_pto_t).astype(int)
  drive_1s = np.repeat(drive_pto_w[:-1], drive_steps) # drive PTO, W per second

  pto_dur = len(stop_1s)   # stop profile length [s]

  #assigning stop pto profiles to waypoints
  is_stop = stop > 0 # true where the vehicle stops
  stop_on = is_stop & (is_on > 0) & (pto_dur > 0) # its a stop & stop pto is on & it has stop profile

  # putting the stop PTO in the center of stops same as VECTO
  pto_start = np.full (len(stop),np.nan) # NaN = no stop profile here
  pto_start[stop_on] = np.floor((stop[stop_on] - pto_dur) / 2) # floor for odd seconds = HDEV assumption

  too_short = np.where(pto_start < 0)[0]
  if len(too_short) > 0:
      raise ValueError(f"PTO ({pto_dur} s) is longer than the stop at waypoint(s) {too_short.tolist()}; the parser must extend these stops")

  #apply the effeciencies
  stop_1s = stop_1s / stop_pto_eff
  drive_1s = drive_1s / drive_pto_eff

  return is_on, pto_start, stop_1s, drive_1s

##CONSUMPTION ENGINE

In [ ]:
def consumption_engine(T, S, V, M, dt,                                     # driver_model rows
                       k_lin, k_roll, k_rot, L, v_avg,                  # force_engine
                       m_vehicle, wp_s, payload_wp, beta,               # mass; payload_wp from payload_engine
                       t_start, T_amb_h, W1, W2, cdxa_v, CdxA0, p_air, R_air,         # weather per hour + crosswind tables
                       ES, PS, HVAC, SP, cond, PTO_always_on,           # P_aux outputs (HVAC scalar or per hour)
                       pto_1s, pto_on_col, pto_start_col,               # pto_engine outputs
                       motor_eff, inverter_eff, gearbox_eff, regen_effeiciency, v_no_regen, P_regen_max=None):

  T, S, V, M = (np.asarray(x) for x in (T, S, V, M))                              # driver rows
  wp_s, payload_wp = np.asarray(wp_s, dtype=float), np.asarray(payload_wp, dtype=float)
  T_amb_h, cdxa_v = np.asarray(T_amb_h, dtype=float), np.asarray(cdxa_v, dtype=float)
  pto_1s, pto_on_col, pto_start_col = (np.asarray(x, dtype=float) for x in (pto_1s, pto_on_col, pto_start_col))

    # --- 1. mass per row: payload forward-filled by distance (payload_engine writes it on start + stop waypoints only)
  has_pl = ~np.isnan(payload_wp)                                   # waypoints where payload is set
  k_pl = np.searchsorted(wp_s[has_pl], S, side="right") - 1        # last payload waypoint at or before each row
  m_row = m_vehicle + payload_wp[has_pl][k_pl]                     # vehicle + payload [kg]

    # --- 2. weather hour of each row; hourly arrays start at hour 0 = t_start reference
  hour = ((t_start + T) // 3600).astype(int)                      # index into T_amb_h, W1, W2, HVAC (if per hour)

    # --- 3. air density and CdxA_eff per row
  rho_row = p_air / (R_air * (T_amb_h[hour] + 273.15))             # ideal gas, fixed pressure (HDEV; VECTO fixed 1.188)

  cdxa_tab = CdxA0 * np.asarray(W1) + np.asarray(W2)               # CdxA_eff table: one row per weather hour x speed grid (VECTO AirdragDataAdapter.cs 59-87)
  v_kmh = np.clip(v_avg * 3.6, cdxa_v[0], cdxa_v[-1])              # hold end values: below 60 km/h uses the 60 value like VECTO
  j = np.clip(np.searchsorted(cdxa_v, v_kmh, side="right") - 1, 0, len(cdxa_v) - 2)  # grid cell of each row
  w = (v_kmh - cdxa_v[j]) / (cdxa_v[j+1] - cdxa_v[j])              # position inside the cell
  cdxa_row = cdxa_tab[hour, j] * (1 - w) + cdxa_tab[hour, j+1] * w  # linear interpolation by speed, at the hour's own wind

    # --- 4. wheel power per row [W]: positive = driving, negative = braking/coasting
  P_wheel = m_row * k_lin + m_row**beta * k_roll + k_rot + rho_row * cdxa_row * L

    # --- 5. wheel power -> electric traction power [W]; positive = drawn, negative = recovered
  drivetrain_effeiciency = motor_eff * inverter_eff * gearbox_eff
  P_trac = np.where(P_wheel > 0, P_wheel / drivetrain_effeiciency, P_wheel * regen_effeiciency)  # Bob's old rule
  P_trac = np.where(M == 3, 0.0, P_trac)                           # coasting/rolling: no motor torque (VECTO BatteryElectricMotorController.cs 44-46)
  P_trac = np.where((M == 4) & (V <= v_no_regen), 0.0, P_trac)     # braking below v_no_regen: brakes only (same file 62-69; 5 km/h Constants.cs 323)
  if P_regen_max is not None:
      P_trac = np.maximum(P_trac, -P_regen_max)                    # regen cap; the rest goes to mechanical brakes


        # --- 6. aux power per row [W]; all inputs already electric W from P_aux / pto_engine
  n = len(T)
  pto_row = np.zeros(n)                                            # stop PTO W per row
  pto_active = np.zeros(n, dtype=bool)                             # True while a stop PTO profile runs
  for i in np.where(pto_on_col == 1)[0]:                           # loop over stops with PTO on (not over rows)
      at_stop = np.where((V == 0) & np.isclose(S, wp_s[i], rtol=0, atol=1e-3))[0]  # standing rows of this stop
      if len(at_stop) == 0:
          raise ValueError(f"no standing rows found for stop at {wp_s[i]} m")       # guard only
      k0 = at_stop[0] + int(pto_start_col[i])                      # PTO start inside the stop (middle, from pto_engine)
      k1 = k0 + len(pto_1s)
      if k1 > at_stop[-1] + 1:
          raise ValueError(f"PTO does not fit in stop at {wp_s[i]} m")             # guard only; parser extends stops
      pto_row[k0:k1] = pto_1s
      pto_active[k0:k1] = True

  moving = V > 0                                                   # SP on only when moving (VECTO ElectricAuxiliaries.cs, ID "STP")
  HVAC_row = np.asarray(HVAC, dtype=float)[hour] if np.ndim(HVAC) else float(HVAC)  # one number, or one per weather hour
  P_aux_row = (ES + PS + HVAC_row                                  # always on
               + SP * moving
               + cond * (moving | pto_active)                      # HDEV rule, see flags
               + PTO_always_on
               + pto_row)

    # --- 7. power at the battery terminals and trip energy
  P_battery = P_trac + P_aux_row                                   # [W]; aux via DC/DC eff 1.0 like VECTO (DeclarationData.cs line 124); battery Ri loss is in the battery engine
  E_battery = np.sum(P_battery * dt) / 3.6e6                       # net trip energy [kWh]

  return P_wheel, P_trac, P_aux_row, P_battery, E_battery